# Inside Bar Breakout on SPY
## Strategy Brief
The Inside Bar Breakout strategy identifies potential breakout opportunities by looking for inside bars, where the high and low of a bar are within the high and low of the previous bar. The prediction is that a breakout from the inside bar's range can lead to a significant price movement. The trade logic involves entering a position when the price breaks above or below the inside bar's range. Historical testing on SPY suggests that this strategy can capture trending moves, though it may be prone to false breakouts.
## References
- https://en.wikipedia.org/wiki/Inside_(2023_film)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the trading context and parameters for the Inside Bar Breakout strategy. We will configure our parameters such as the stock ticker, the lookback period for identifying inside bars, and the initial capital for backtesting.

In [ ]:
TICKER = 'SPY'
START_DATE = '2010-01-01'
LOOKBACK_PERIOD = 1
INITIAL_CAPITAL = 10000

## PHASE 2 - Data Exploration
In this phase, we download historical price data for SPY using yfinance and compute the necessary indicators for the Inside Bar Breakout strategy. We will then plot these indicators overlaid on the price chart to visualize potential signals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE)

# Compute inside bar indicators
data['Prev_High'] = data['High'].shift(1)
data['Prev_Low'] = data['Low'].shift(1)
data['Inside_Bar'] = (data['High'] < data['Prev_High']) & (data['Low'] > data['Prev_Low'])

# Plot price and inside bars
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.scatter(data.index, data['Close'][data['Inside_Bar']], color='red', label='Inside Bar', marker='o')
plt.title('SPY Price and Inside Bars')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
In this phase, we create the signal for the Inside Bar Breakout strategy. We define the entry and exit logic based on the breakout of the inside bar's range and compute the positions series to indicate when we are in a trade.

In [ ]:
# Create signals
data['Signal'] = 0
breakout_up = (data['Close'] > data['Prev_High']) & data['Inside_Bar']
breakout_down = (data['Close'] < data['Prev_Low']) & data['Inside_Bar']
data.loc[breakout_up, 'Signal'] = 1
data.loc[breakout_down, 'Signal'] = -1

# Compute positions
data['Position'] = data['Signal'].replace(to_replace=0, method='ffill').shift(1)

## PHASE 4 - Coding & Backtesting
In this phase, we backtest the strategy using the positions series. We calculate daily returns and plot the equity curve to visualize the strategy's performance over time.

In [ ]:
# Calculate daily returns
data['Market_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Position'].shift(1) * data['Market_Return']

data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod() * INITIAL_CAPITAL

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve of Inside Bar Breakout Strategy')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
In this phase, we evaluate the performance of the strategy using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We also compare the strategy's performance against a buy-and-hold approach.

In [ ]:
def calculate_performance(data):
    # CAGR
    total_return = data['Equity_Curve'].iloc[-1] / INITIAL_CAPITAL - 1
    years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1
    
    # Sharpe Ratio
    sharpe_ratio = data['Strategy_Return'].mean() / data['Strategy_Return'].std() * np.sqrt(252)
    
    # Sortino Ratio
    downside_std = data['Strategy_Return'][data['Strategy_Return'] < 0].std()
    sortino_ratio = data['Strategy_Return'].mean() / downside_std * np.sqrt(252)
    
    # Calmar Ratio
    max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max() / data['Equity_Curve'].cummax().max()
    calmar_ratio = cagr / max_drawdown
    
    # Buy and Hold
    buy_and_hold_return = data['Close'].iloc[-1] / data['Close'].iloc[0] - 1
    
    # Performance table
    performance = pd.DataFrame({
        'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown', 'Buy and Hold Return'],
        'Strategy': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown, total_return],
        'Buy and Hold': [buy_and_hold_return, None, None, None, None, buy_and_hold_return]
    })
    return performance

performance = calculate_performance(data)
print(performance)

## PHASE 6 - Deploy & Monitor
In this phase, we create a function to download the last 60 days of SPY data, compute today's signal, and print the current position. This function can be used to monitor the strategy in real-time.

In [ ]:
def monitor_strategy():
    recent_data = yf.download(TICKER, period='60d')
    recent_data['Prev_High'] = recent_data['High'].shift(1)
    recent_data['Prev_Low'] = recent_data['Low'].shift(1)
    recent_data['Inside_Bar'] = (recent_data['High'] < recent_data['Prev_High']) & (recent_data['Low'] > recent_data['Prev_Low'])
    
    recent_data['Signal'] = 0
    breakout_up = (recent_data['Close'] > recent_data['Prev_High']) & recent_data['Inside_Bar']
    breakout_down = (recent_data['Close'] < recent_data['Prev_Low']) & recent_data['Inside_Bar']
    recent_data.loc[breakout_up, 'Signal'] = 1
    recent_data.loc[breakout_down, 'Signal'] = -1
    
    recent_data['Position'] = recent_data['Signal'].replace(to_replace=0, method='ffill').shift(1)
    
    print(f"Today's Position: {recent_data['Position'].iloc[-1]}")

monitor_strategy()